# Actual workflow correction recorded 23 October 2026

Historical sections below are designs, not evidence that every proposed step happened. Magnus confirms: 24 h nutrient-rich preculture, OD-based dilution **without cell washing**, Echo transfer into crude-glycerol medium, 48 h incubation, spin at centrifuge setting 0 °C, 5 µL supernatant + 10 µL crude-glycerol medium, 15 µL assay mix, 1 h light-protected incubation, 15 µL 1 M acetic acid stop, then absorbance. No additional sample cleanup.

Use the concise culture and assay SOPs in `../` for the corrected procedure. The original planning recipes are archived unchanged. Reader wavelength is provisionally 490 nm from the SOP; the October reader CSV headers do not independently confirm it. OD/pH/extra-control templates are not measurements. Read [the independent-analysis handoff](../../docs/handoff/START_HERE.md). Do not Run All to reconstruct a historical experiment.


# Archived run 1 — 2026-08-28 — 13 resolved strains

This notebook builds one 384-well destination plate using only the resolved members:

- nr1–nr12, with nr12 / lab marking `296` controlled by `INCLUDE_296`
- nr14 / lab marking `264` included
- unresolved nr13 (`P2_D9`) always excluded

The plate uses 306 interior wells (`B2:O23`) and leaves the complete edge ring plus two interior wells unused. It contains 102 unique communities with three technical replicates each. All monocultures, all leave-one-out communities, and the full community are guaranteed; the remaining communities are spread as evenly as possible over intermediate sizes and chosen with a deterministic farthest-point design to maximize compositional diversity. Replicates are placed in three separated regions of the plate.

## Run workflow

1. Set `INCLUDE_296` after checking whether 296 grew.
2. Enter the 1:10 plate-reader values. The preparation table converts them to actual OD600 and gives a 50 µL OD600 1.0 working-suspension recipe. Cultures below OD600 1.0 remain undiluted.
3. Verify the source-well map for source plate `396`. The defaults are the resolved library wells from `Strain Identity and Lab Mapping.md`.
4. Run all cells. Export occurs only when every active OD is valid and every well stays at or below 2,000 nL total inoculum.


In [ ]:
from pathlib import Path
from datetime import date
from math import comb
import itertools
import string

import numpy as np
import pandas as pd

# ---------------------------- run configuration ----------------------------
INCLUDE_296 = True  # change to False if nr12 / lab marking 296 did not grow
SOURCE_PLATE_NAME = "396"
DESTINATION_PLATE_NAME = "community_plate_01"
RANDOM_SEED = 20260828
RUN_LABEL = "20260828"  # frozen: this section is the traceable first-run setup
OUTPUT_DIR = Path("experiment_setup_output")

TARGET_MEMBER_BIOMASS_OD_UL = 0.100  # gives 100 nL when source OD600 is 1.0
MAX_TOTAL_INOCULUM_NL = 2000.0
ECHO_DROPLET_NL = 25.0  # one Echo droplet; all executed volumes must be divisible by 25 nL
TECHNICAL_REPLICATES = 3


## 1. Resolved strain panel and source wells

`source_well` is initialized from the resolved library mapping. If plate 396 is loaded in a different pattern, edit those coordinates before generating the picklist.


In [ ]:
panel = pd.DataFrame([
    {"member_nr": 1,  "lab_marking": "256", "assembly_id": "P2_C2",  "source_well": "I8"},
    {"member_nr": 2,  "lab_marking": "75",  "assembly_id": "P3_D2",  "source_well": "D1"},
    {"member_nr": 3,  "lab_marking": "269", "assembly_id": "P2_C10", "source_well": "J21"},
    {"member_nr": 4,  "lab_marking": "294", "assembly_id": "P3_G8",  "source_well": "I20"},
    {"member_nr": 5,  "lab_marking": "275", "assembly_id": "P3_C3",  "source_well": "L1"},
    {"member_nr": 6,  "lab_marking": "41",  "assembly_id": "P1_B11", "source_well": "B17"},
    {"member_nr": 7,  "lab_marking": "158", "assembly_id": "P1_F7",  "source_well": "E16"},
    {"member_nr": 8,  "lab_marking": "145", "assembly_id": "P3_F11", "source_well": "E21"},
    {"member_nr": 9,  "lab_marking": "200", "assembly_id": "P3_A9",  "source_well": "G6"},
    {"member_nr": 10, "lab_marking": "196", "assembly_id": "P3_A8",  "source_well": "G2"},
    {"member_nr": 11, "lab_marking": "291", "assembly_id": "P3_C5",  "source_well": "K13"},
    {"member_nr": 12, "lab_marking": "296", "assembly_id": "P3_G9",  "source_well": "I22"},
    {"member_nr": 14, "lab_marking": "264", "assembly_id": "P2_B6",  "source_well": "I14"},
])

unresolved = pd.DataFrame([
    {"member_nr": 13, "lab_marking": pd.NA, "assembly_id": "P2_D9", "source_well": pd.NA}
])

active_panel = panel.loc[INCLUDE_296 | panel["lab_marking"].ne("296")].copy()
active_panel = active_panel.sort_values("member_nr").reset_index(drop=True)

expected_n = 13 if INCLUDE_296 else 12
assert len(active_panel) == expected_n
assert active_panel["source_well"].notna().all()
assert active_panel["source_well"].is_unique
assert 13 not in set(active_panel["member_nr"])
assert ("296" in set(active_panel["lab_marking"])) == INCLUDE_296

print(f"Scenario: {len(active_panel)} resolved strains; INCLUDE_296={INCLUDE_296}")
display(active_panel)
print("Excluded unresolved member:")
display(unresolved)


## 2. Prepare OD600 1.0 working suspensions

The readings below were measured at 1:10, so actual culture OD600 is `reading × 10`. For every culture above OD600 1.0, the table gives a direct dilution recipe for 50 µL working suspension at OD600 1.0. Load the complete 50 µL preparation into the listed source well. Cultures below OD600 1.0 are used undiluted; their Echo transfer is adjusted instead of concentrating them.


In [ ]:
OD600_READING_AT_1_TO_10 = {
    "256": 0.99,
    "75":  0.75,
    "269": 0.08,
    "294": 0.98,
    "275": 1.33,
    "41":  0.71,
    "158": 1.07,
    "145": 1.22,
    "200": 0.32,
    "196": 0.09,
    "291": 0.15,
    "296": 0.16,
    "264": 0.81,
}
READING_DILUTION_FACTOR = 10.0
TARGET_WORKING_OD600 = 1.0
WORKING_PREP_VOLUME_UL = 50.0  # also the source-well load volume

source_measurements = active_panel.copy()
source_measurements["OD600_reading_at_1_to_10"] = source_measurements["lab_marking"].map(OD600_READING_AT_1_TO_10)
source_measurements["actual_culture_OD600"] = source_measurements["OD600_reading_at_1_to_10"] * READING_DILUTION_FACTOR
source_measurements["culture_uL_for_working_prep"] = np.where(
    source_measurements["actual_culture_OD600"] > TARGET_WORKING_OD600,
    TARGET_WORKING_OD600 * WORKING_PREP_VOLUME_UL / source_measurements["actual_culture_OD600"],
    WORKING_PREP_VOLUME_UL,
)
source_measurements["diluent_uL_for_working_prep"] = (
    WORKING_PREP_VOLUME_UL - source_measurements["culture_uL_for_working_prep"]
)
source_measurements["measured_OD600"] = (
    source_measurements["actual_culture_OD600"]
    * source_measurements["culture_uL_for_working_prep"]
    / WORKING_PREP_VOLUME_UL
)
source_measurements["dilution_factor"] = (
    WORKING_PREP_VOLUME_UL / source_measurements["culture_uL_for_working_prep"]
)

prep_columns = [
    "member_nr", "lab_marking", "source_well", "actual_culture_OD600",
    "dilution_factor", "culture_uL_for_working_prep",
    "diluent_uL_for_working_prep", "measured_OD600",
]
display(source_measurements[prep_columns].round(2))


## 3. Build a maximally varied interior-well community design

The design places 102 unique communities in triplicate (306 wells) within `B2:O23` and never uses an edge well. It guarantees every monoculture, every leave-one-out community, and the full community. Remaining capacity is balanced across sizes 2 through N−2. Within each size, a greedy maximin Hamming-distance selection spreads compositions apart; member-use balance breaks ties. The three replicates of each community are assigned to positions separated by 103 places in the row-major interior-well list.


In [ ]:
ROWS_384 = list(string.ascii_uppercase[:16])
COLS_384 = list(range(1, 25))
INTERIOR_WELLS = [f"{row}{col}" for row in ROWS_384[1:-1] for col in COLS_384[1:-1]]
assert len(INTERIOR_WELLS) == 14 * 22 == 308

def allocate_size_quotas(n_members, n_wells):
    capacities = {k: comb(n_members, k) for k in range(1, n_members + 1)}
    quotas = {k: 0 for k in capacities}

    # Anchor the design at both extremes.
    for k in (1, n_members - 1, n_members):
        quotas[k] = capacities[k]

    remaining = n_wells - sum(quotas.values())
    middle_sizes = list(range(2, n_members - 1))
    assert remaining >= 0

    # Water-fill the intermediate sizes as evenly as their capacities allow.
    while remaining:
        progressed = False
        for k in middle_sizes:
            if quotas[k] < capacities[k] and remaining:
                quotas[k] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            raise ValueError("Requested more unique communities than exist.")
    return quotas

def select_diverse_subsets(member_labels, size, take, rng):
    candidates = list(itertools.combinations(member_labels, size))
    if take == len(candidates):
        return candidates

    member_index = {member: i for i, member in enumerate(member_labels)}
    matrix = np.zeros((len(candidates), len(member_labels)), dtype=np.int8)
    for row_i, candidate in enumerate(candidates):
        matrix[row_i, [member_index[m] for m in candidate]] = 1

    first = int(rng.integers(len(candidates)))
    selected = [first]
    available = np.ones(len(candidates), dtype=bool)
    available[first] = False
    member_counts = matrix[first].astype(int)
    nearest_distance = np.abs(matrix - matrix[first]).sum(axis=1)

    while len(selected) < take:
        best_distance = nearest_distance[available].max()
        eligible = np.flatnonzero(available & (nearest_distance == best_distance))
        reuse_scores = matrix[eligible] @ member_counts
        eligible = eligible[reuse_scores == reuse_scores.min()]
        chosen = int(rng.choice(eligible))

        selected.append(chosen)
        available[chosen] = False
        member_counts += matrix[chosen]
        distance_to_chosen = np.abs(matrix - matrix[chosen]).sum(axis=1)
        nearest_distance = np.minimum(nearest_distance, distance_to_chosen)

    return [candidates[i] for i in selected]

members = active_panel["lab_marking"].tolist()
n_members = len(members)
n_unique_communities = len(INTERIOR_WELLS) // TECHNICAL_REPLICATES
n_used_wells = n_unique_communities * TECHNICAL_REPLICATES
size_quotas = allocate_size_quotas(n_members, n_unique_communities)
selection_rng = np.random.default_rng(RANDOM_SEED)

communities = []
for size in range(1, n_members + 1):
    communities.extend(select_diverse_subsets(members, size, size_quotas[size], selection_rng))

assert len(communities) == n_unique_communities
assert len(set(communities)) == len(communities)
assert sum(len(c) == 1 for c in communities) == n_members
assert sum(len(c) == n_members for c in communities) == 1

communities = sorted(communities, key=lambda c: (len(c), c))
conditions = pd.DataFrame({
    "community_id": [f"C{i:03d}" for i in range(1, len(communities) + 1)],
    "community_size": [len(c) for c in communities],
    "members": ["|".join(c) for c in communities],
})

# Three 102-well blocks separated by one unused well give each condition
# spatially separated technical replicates. Randomize condition-to-slot mapping.
condition_order = selection_rng.permutation(n_unique_communities)
replicate_offsets = [0, n_unique_communities + 1, 2 * (n_unique_communities + 1)]
layout_rows = []
for slot, condition_i in enumerate(condition_order):
    condition = conditions.iloc[condition_i]
    replicate_labels = selection_rng.permutation(np.arange(1, TECHNICAL_REPLICATES + 1))
    for block_i, offset in enumerate(replicate_offsets):
        layout_rows.append({
            "Destination Plate Name": DESTINATION_PLATE_NAME,
            "Destination Well": INTERIOR_WELLS[slot + offset],
            "technical_replicate": int(replicate_labels[block_i]),
            "community_id": condition["community_id"],
            "community_size": int(condition["community_size"]),
            "members": condition["members"],
        })
design = pd.DataFrame(layout_rows)
unused_interior_wells = sorted(set(INTERIOR_WELLS) - set(design["Destination Well"]))

for member in members:
    design[f"member_{member}"] = design["members"].str.split("|").apply(lambda x: int(member in x))

edge_wells = {f"{r}{c}" for r in ROWS_384 for c in COLS_384 if r in ("A", "P") or c in (1, 24)}
assert set(design["Destination Well"]).isdisjoint(edge_wells)
assert design["Destination Well"].is_unique
assert len(design) == n_used_wells == 306
assert len(unused_interior_wells) == 2
assert design.groupby("community_id").size().eq(TECHNICAL_REPLICATES).all()
assert design.groupby("community_id")["Destination Well"].nunique().eq(TECHNICAL_REPLICATES).all()
assert design.groupby("community_id")["technical_replicate"].nunique().eq(TECHNICAL_REPLICATES).all()
well_coordinates = {f"{r}{c}": (ri, ci) for ri, r in enumerate(ROWS_384) for ci, c in enumerate(COLS_384)}
replicate_distances = []
for _, replicate_group in design.groupby("community_id"):
    coordinates = [well_coordinates[w] for w in replicate_group["Destination Well"]]
    replicate_distances.extend(
        abs(coordinates[a][0] - coordinates[b][0]) + abs(coordinates[a][1] - coordinates[b][1])
        for a, b in itertools.combinations(range(TECHNICAL_REPLICATES), 2)
    )
assert min(replicate_distances) >= 11

quota_table = pd.DataFrame({
    "community_size": list(size_quotas),
    "selected_unique_communities": list(size_quotas.values()),
    "possible_unique_communities": [comb(n_members, k) for k in size_quotas],
})
display(quota_table)
display(design.head(20))
print(f"Designed {n_unique_communities} unique communities × {TECHNICAL_REPLICATES} technical replicates = {len(design)} wells.")
print(f"Unused interior wells: {', '.join(unused_interior_wells)}")


In [ ]:
# Plate map: values are community sizes; dashes are the unused edge ring.
plate_values = np.full((16, 24), np.nan)
well_to_position = {f"{r}{c}": (ri, ci) for ri, r in enumerate(ROWS_384) for ci, c in enumerate(COLS_384)}
for row in design.itertuples(index=False):
    ri, ci = well_to_position[getattr(row, "_1")]  # Destination Well
    plate_values[ri, ci] = row.community_size

plate_map = pd.DataFrame(plate_values, index=ROWS_384, columns=COLS_384)
plate_map = plate_map.map(lambda value: "–" if pd.isna(value) else str(int(value)))
print(f"{DESTINATION_PLATE_NAME}: each value is community size ({n_members}-strain scenario)")
display(plate_map)


## 4. Dose-normalized Echo picklist

Every species receives the same nominal biomass dose of 0.100 OD600·µL, independent of community size:

`calculated transfer per member = 100 / measured working-source OD600 nL`

`Transfer Volume` is the total requested volume in nL, rounded to the nearest 25 nL droplet. Every exported value is therefore divisible by 25. The audit table records calculated and executed volumes and realized biomass. Picklist export is blocked if an active OD is missing/non-positive or if a well would exceed 2,000 nL total inoculum.


In [ ]:
def round_to_droplet(volume_nl, droplet_nl=ECHO_DROPLET_NL):
    return np.floor(volume_nl / droplet_nl + 0.5) * droplet_nl

od_lookup = source_measurements.set_index("lab_marking")["measured_OD600"].to_dict()
source_well_lookup = active_panel.set_index("lab_marking")["source_well"].to_dict()
invalid_ods = {lab: od_lookup.get(lab) for lab in members if pd.isna(od_lookup.get(lab)) or od_lookup.get(lab) <= 0}

picklist_ready = False
transfer_audit = pd.DataFrame()
well_audit = pd.DataFrame()
echo_picklist = pd.DataFrame()
source_demand = pd.DataFrame()

if invalid_ods:
    print("PICKLIST NOT GENERATED — enter a positive OD600 for: " + ", ".join(invalid_ods))
else:
    transfer_rows = []
    for destination in design.itertuples(index=False):
        destination_well = getattr(destination, "_1")  # Destination Well
        community_members = destination.members.split("|")
        k = destination.community_size
        target_per_member = TARGET_MEMBER_BIOMASS_OD_UL

        for lab_marking in community_members:
            measured_od = float(od_lookup[lab_marking])
            calculated_nl = 1000.0 * target_per_member / measured_od
            executed_nl = int(round_to_droplet(calculated_nl))
            transfer_rows.append({
                "Source Plate Name": SOURCE_PLATE_NAME,
                "Source Well": source_well_lookup[lab_marking],
                "Destination Plate Name": DESTINATION_PLATE_NAME,
                "Destination Well": destination_well,
                "Transfer Volume": executed_nl,
                "community_id": destination.community_id,
                "technical_replicate": destination.technical_replicate,
                "community_size": k,
                "lab_marking": lab_marking,
                "measured_OD600": measured_od,
                "target_member_OD600_uL": target_per_member,
                "calculated_transfer_nL": calculated_nl,
                "executed_transfer_nL": executed_nl,
                "realized_member_OD600_uL": measured_od * executed_nl / 1000.0,
            })

    transfer_audit = pd.DataFrame(transfer_rows)
    assert (transfer_audit["executed_transfer_nL"] % ECHO_DROPLET_NL == 0).all(), "non-droplet transfer found"
    well_audit = (
        transfer_audit.groupby(["Destination Plate Name", "Destination Well", "community_id", "technical_replicate", "community_size"], as_index=False)
        .agg(
            total_inoculum_nL=("executed_transfer_nL", "sum"),
            realized_total_OD600_uL=("realized_member_OD600_uL", "sum"),
        )
    )
    well_audit["target_total_OD600_uL"] = TARGET_MEMBER_BIOMASS_OD_UL * well_audit["community_size"]
    well_audit["biomass_error_pct"] = 100 * (
        well_audit["realized_total_OD600_uL"] / well_audit["target_total_OD600_uL"] - 1
    )

    volume_violations = well_audit.loc[well_audit["total_inoculum_nL"] > MAX_TOTAL_INOCULUM_NL + 1e-9]
    zero_transfers = transfer_audit.loc[transfer_audit["executed_transfer_nL"] <= 0]

    if len(volume_violations) or len(zero_transfers):
        print("PICKLIST NOT READY — concentrate the relevant source suspension(s), update OD600, and rerun.")
        if len(volume_violations):
            print(f"{len(volume_violations)} destination wells exceed {MAX_TOTAL_INOCULUM_NL:.0f} nL:")
            display(volume_violations)
        if len(zero_transfers):
            print("Transfers rounded to zero:")
            display(zero_transfers)
    else:
        echo_columns = ["Source Plate Name", "Source Well", "Destination Plate Name", "Destination Well", "Transfer Volume"]
        echo_picklist = transfer_audit[echo_columns].copy()
        assert (echo_picklist["Transfer Volume"] % ECHO_DROPLET_NL == 0).all()
        source_demand = (
            transfer_audit.groupby(["lab_marking", "Source Well"], as_index=False)
            .agg(number_of_destination_wells=("Destination Well", "nunique"), total_transfer_nL=("executed_transfer_nL", "sum"))
            .sort_values("lab_marking")
        )
        picklist_ready = True
        print(f"PICKLIST READY: {len(echo_picklist)} transfers into {len(well_audit)} wells.")
        print(f"Maximum total inoculum: {well_audit['total_inoculum_nL'].max():.1f} nL.")
        print(f"Maximum absolute biomass rounding error: {well_audit['biomass_error_pct'].abs().max():.3f}%.")
        display(echo_picklist.head(20))
        display(source_demand)


## 5. Export

Exports are scenario-labelled so the 12- and 13-strain picklists cannot be confused. `source_demand` is transferred volume only; add the validated dead volume for plate loading.


In [ ]:
scenario = f"{n_members}strains" if INCLUDE_296 else f"{n_members}strains_no296"
stem = f"{RUN_LABEL}_{scenario}"

if not picklist_ready:
    print("EXPORT SKIPPED — complete/fix the OD600 inputs and rerun the picklist cell.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    active_panel.to_csv(OUTPUT_DIR / f"source_plate_map_{stem}.csv", index=False)
    source_measurements[prep_columns].to_csv(OUTPUT_DIR / f"dilution_plan_{stem}.csv", index=False)
    design.to_csv(OUTPUT_DIR / f"community_design_{stem}.csv", index=False)
    echo_picklist.to_csv(OUTPUT_DIR / f"echo_picklist_{stem}.csv", index=False)
    transfer_audit.to_csv(OUTPUT_DIR / f"transfer_audit_{stem}.csv", index=False)
    well_audit.to_csv(OUTPUT_DIR / f"well_audit_{stem}.csv", index=False)
    source_demand.to_csv(OUTPUT_DIR / f"source_demand_{stem}.csv", index=False)

    print(f"Saved seven files to {OUTPUT_DIR.resolve()}")
    for path in sorted(OUTPUT_DIR.glob(f"*{stem}.csv")):
        print(path.name)


# Adaptive run 2 — 2026-09-02

This section is appended rather than replacing run 1. It loads the **2026-08-30 wiped-bottom rerun**, joins it to the frozen 2026-08-28 community design, uses the median of each set of three technical replicates, subtracts the median of the same-medium edge blanks (excluding standards A1:F1), and fits an exact Gaussian-process Bayesian surrogate over the 13 binary species indicators. The response is therefore **relative assay absorbance**, not an asserted absolute lactate concentration.

Four current plates are generated, each with 102 unique communities and three technical replicates in non-edge wells:

1. repeat/backfill of the previous maximally diverse design, retaining the exact community and well layout wherever every member is available;
2. Bayesian expected improvement;
3. Bayesian posterior mean;
4. uniform random sampling from the remaining candidate compositions.

Communities are deduplicated across all four current strategies. The repeat plate is the only intentional reuse of a previously measured community. If a species is excluded, feasible previous communities remain exact repeats in their original wells and impossible repeat slots receive same-richness maximin-Hamming backfills over the active panel. `repeat_status` records which is which. Deduplication priority is repeat/backfill → EI → posterior mean → random, so the random batch cannot consume a top model proposal. Candidate predictions retain both raw acquisition ranks and final strategy assignments for audit.

## Bench dosing rule

The source plate is always one contiguous row in the active-species order. For 2026-09-02, `296` is excluded after a second no-growth result, so `A1:A12` contains `256, 75, 269, 294, 275, 41, 158, 145, 200, 196, 291, 264`. Its previous assay data remain in GP training, but no current candidate or transfer can contain it. Prepare at most 50 µL per well, targeting OD600 1.20 when dilution is possible; weaker cultures remain undiluted.

The preferred mode fixes total starting biomass at `0.100 × active species count` OD600·µL per destination well: **1.200 OD600·µL for today's 12-species panel**. Thus the full community receives approximately 0.100 OD600·µL per species, corresponding to roughly 100 nL/species at OD600 1.0. Echo transfers are rounded to 25 nL. It is used only if every 50 µL source fill retains at least **15 µL dead volume** after dispensing its plate. Otherwise the code automatically falls back to the prior fixed-per-member dose of 0.100 OD600·µL, so richer communities receive more total starting biomass, and records that fact.

Because four plates require more than one 50 µL source fill, each plate receives its own picklist. **Refill source row A between destination plates.** The source-demand audit reports transferred volume before instrument dead volume.


In [ ]:
# Historical OD inputs remain explicit here so the run history is self-contained.
OD600_READING_AT_1_TO_10_BY_RUN = {
    "20260828": {
        "256": 0.99, "75": 0.75, "269": 0.08, "294": 0.98,
        "275": 1.33, "41": 0.71, "158": 1.07, "145": 1.22,
        "200": 0.32, "196": 0.09, "291": 0.15, "296": 0.16,
        "264": 0.81,
    },
    "20260902": {
        # Today's 1:10 plate-reader values, in the declared source-row order.
        "256": 0.53, "75": 0.61, "269": 0.06, "294": 0.28,
        "275": 1.22, "41": 0.58, "158": 0.81, "145": 0.53,
        "200": 0.22, "196": 0.08, "291": 0.13, "296": 0.00,
        "264": 0.76,
    },
}

ADAPTIVE_RUN_LABEL = "20260902"
ADAPTIVE_EXCLUDED_MEMBERS = {"296"}  # second no-growth result; retain only in training history
ADAPTIVE_RANDOM_SEED = 20260902
ADAPTIVE_SOURCE_PLATE_NAME = "396"
ADAPTIVE_SOURCE_PREP_UL = 50.0
ADAPTIVE_TARGET_WORKING_OD600 = 1.20
ADAPTIVE_SOURCE_DEAD_VOLUME_UL = 15.0
ADAPTIVE_MAX_DESTINATION_INOCULUM_UL = 15.0
ADAPTIVE_EI_XI = 0.002


In [ ]:
import sys

echo_dir = Path.cwd()
if not (echo_dir / "adaptive_run.py").exists():
    echo_dir = Path("glycerol_lactate_community/protocols/echo").resolve()
if str(echo_dir) not in sys.path:
    sys.path.insert(0, str(echo_dir))

from adaptive_run import run_adaptive_design

adaptive = run_adaptive_design(
    run_label=ADAPTIVE_RUN_LABEL,
    od_readings=OD600_READING_AT_1_TO_10_BY_RUN[ADAPTIVE_RUN_LABEL],
    base_dir=echo_dir,
    random_seed=ADAPTIVE_RANDOM_SEED,
    ei_xi=ADAPTIVE_EI_XI,
    prep_volume_ul=ADAPTIVE_SOURCE_PREP_UL,
    target_working_od=ADAPTIVE_TARGET_WORKING_OD600,
    source_dead_volume_ul=ADAPTIVE_SOURCE_DEAD_VOLUME_UL,
    max_destination_inoculum_ul=ADAPTIVE_MAX_DESTINATION_INOCULUM_UL,
    source_plate_name=ADAPTIVE_SOURCE_PLATE_NAME,
    excluded_members=ADAPTIVE_EXCLUDED_MEMBERS,
    export=True,
)

print("Bayesian model and previous-run diagnostics")
display(adaptive["model_summary"].round(4))
print("Four deduplicated sampling strategies")
display(adaptive["strategy_summary"])
print("Today's one-row source map and OD/dilution input")
display(adaptive["source_plan"].round(3))
print(adaptive["picklist_message"])
print(f"Outputs: {adaptive['output_dir']}")

selected_preview = (
    adaptive["candidate_predictions"]
    .loc[lambda frame: frame["selected_strategy"].ne("")]
    .sort_values(["selected_strategy", "selection_rank"])
)
display(selected_preview.head(25))


# Acquisition round 3 — 2026-09-09 — four separate paths

Run **this appended section** for today's experiment. It is self-contained; the preceding sections remain historical records of the August 28 and September 2 setups, including their original OD inputs. Running historical export cells again would regenerate those earlier outputs.

## Training and selection

The user selected **ordinary/single measurement readouts** for both previous rounds. The response is each community's triplicate median minus the median of **all 384 readings on its own plate**, following the September 8 comparison. This is a relative assay signal. It is not a lactate concentration or a correction for spatial gradients.

- Shared base for both models: August 28 max-diversity communities, read August 30, plus September 2 max-diversity/repeat/backfill communities, read September 4.
- EI model: shared base + **only the EI path's** September 4 observations.
- TM / posterior-mean model: shared base + **only the posterior-mean path's** September 4 observations.
- Every run-level observation is retained. Repeated compositions enter the GP as the median across their run-level responses. Each model has 306 run observations covering 252 unique compositions.
- Random outcomes do not train either model. Max-diversity selection continues from its own historical compositions; random continues uniformly over the remaining unseen compositions.
- All prior compositions from all four paths are excluded from new selection. Each current path gets 102 communities in triplicate, using 306 interior wells on one plate.
- Current cross-path deduplication retains the previous priority: **max-diversity → EI → TM → random**. Full predictions and pre-dedup top proposals are saved, including which proposals were displaced. This priority and one plate per path limit causal comparisons between strategies.

## Bench settings

Plate 1 = EI; plate 2 = continued max-diversity; plate 3 = TM/posterior mean; plate 4 = random. All 76 edge wells and the two unused interior wells F16/K9 receive no bacterial transfer.

Source plate `396` uses a contiguous row A in the established species order. With 13 active species, 296 is **A12** and 264 is **A13**. If 296 is excluded, 264 moves to **A12**. Always use the source map in today's selected scenario.

Enter fresh **1:10 reader ODs** below; actual culture OD is the entered value × 10. `INCLUDE_296_20260909 = "auto"` interprets a positive 296 reading as available, zero as excluded, and a missing reading as pending. No arbitrary positive-OD threshold is imposed. Set it to False explicitly if a positive reading is not usable culture.

Each source fill is at most **50 µL**, diluted toward OD600 **1.20** where possible; weaker cultures remain undiluted. Keep the September 2 executed dose: **0.100 OD600·µL per species**, equivalent to 100 nL at OD 1, across every path and community size. Total biomass therefore increases with richness. **No 15 µL destination cap, no 15 µL source-reserve rule, and no automatic switch to equal-total biomass.** Transfers round to 25 nL; no minimum transfer volume is added. Zero-rounded transfers or demand greater than the actual loaded liquid prevent Echo export without changing the dose. Source remainder is reported for inspection, not certified as within the instrument's ejectable range.

Destination prefill: **50 µL media**, plus the calculated bacterial transfers. The well audit includes final volume. Each picklist assumes a source row restored to 50 µL per well before its plate; do not add another 50 µL on top of residual liquid.

Every execution creates a new timestamped snapshot with OD entries, input hashes, model parameters, exact training rows, candidate scores and audits. Pending OD or pending 296 creates labelled design previews but **no executable Echo picklists**.


In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

# New dated entry; previous run OD dictionaries above remain unchanged.
OD600_READING_AT_1_TO_10_20260909 = {
    "256": 0.60,
    "75":  0.38,
    "269": 0.08,
    "294": 0.87,
    "275": 1.19,
    "41":  0.34,
    "158": 0.50,
    "145": 0.62,
    "200": 0.30,
    "196": 0.09,
    "291": 0.10,
    "296": 0.00,  # no growth: excluded automatically
    "264": 0.44,
}

INCLUDE_296_20260909 = "auto"
READOUT_MODE_20260909 = "single"  # user-confirmed ordinary readouts for both rounds
SEED_20260909 = 20260909
SOURCE_PLATE_20260909 = "396"
SOURCE_LOAD_UL_20260909 = 50.0
TARGET_WORKING_OD_20260909 = 1.20
MEDIA_PREFILL_UL_20260909 = 50.0
EI_XI_20260909 = 0.002

# Locate the notebook directory whether launched from echo/, its project or an ancestor.
echo_dir_20260909 = next(
    (candidate for parent in [Path.cwd(), *Path.cwd().parents]
     for candidate in [parent, parent / "glycerol_lactate_community/protocols/echo"]
     if (candidate / "acquisition_round_20260909.py").is_file()),
    None,
)
if echo_dir_20260909 is None:
    raise FileNotFoundError("Run from the project or protocols/echo directory.")
if str(echo_dir_20260909) not in sys.path:
    sys.path.insert(0, str(echo_dir_20260909))
import adaptive_run
import acquisition_round_20260909
importlib.reload(adaptive_run)
importlib.reload(acquisition_round_20260909)


In [2]:
round3_20260909 = adaptive_run.run_acquisition_round_20260909(
    od_readings=OD600_READING_AT_1_TO_10_20260909,
    include_296=INCLUDE_296_20260909,
    readout_mode=READOUT_MODE_20260909,
    seed=SEED_20260909,
    source_plate_name=SOURCE_PLATE_20260909,
    source_load_ul=SOURCE_LOAD_UL_20260909,
    target_working_od=TARGET_WORKING_OD_20260909,
    medium_ul=MEDIA_PREFILL_UL_20260909,
    xi=EI_XI_20260909,
    base_dir=echo_dir_20260909,
    export=True,
)
print("Two models with separate acquisition histories:")
display(round3_20260909["model_summary"].round(4))
print("LOO diagnostics condition on fitted hyperparameters; they are not prospective validation.")
print("New immutable snapshot:", round3_20260909["output_dir"])

for scenario, data in round3_20260909["scenarios"].items():
    print("\n" + scenario + ": " + data["status"])
    display(data["summary"])
    print("Source row and today's 50 µL preparation:")
    display(data["source_plan"].round(3))
    if len(data["source_demand"]):
        print("Source demand and liquid remaining per destination plate:")
        display(data["source_demand"].round(3))
    if len(data["well_audit"]):
        display(data["well_audit"].groupby("Source Strategy", as_index=False).agg(
            dosing_mode=("dosing_mode", "first"),
            max_inoculum_nL=("total_inoculum_nL", "max"),
            max_final_volume_uL=("final_culture_volume_uL", "max"),
            max_abs_biomass_error_pct=("biomass_error_pct", lambda x: x.abs().max()),
        ).round(3))
    print("Highest-ranked selected proposals from each path:")
    selected = data["candidates"].loc[lambda frame: frame["selected_path"].ne("")]
    display(selected.sort_values(["selected_path", "selection_rank"]).groupby(
        "selected_path", sort=False
    ).head(5)[[
        "selected_path", "selection_rank", "community_size", "members",
        "ei_expected_improvement", "ei_posterior_mean", "tm_posterior_mean"
    ]])


Two models with separate acquisition histories:
                model_path  shared_observations  own_acquired_observations  total_run_observations  unique_training_compositions  \
0              bayesian_ei                  204                        102                     306                           252   
1  bayesian_posterior_mean                  204                        102                     306                           252   

                                 response_definition                                          loo_scope  n_training_communities  length_scale  \
0  median across runs of triplicate median minus ...  conditional on hyperparameters selected using ...                   252.0           1.5   
1  median across runs of triplicate median minus ...  conditional on hyperparameters selected using ...                   252.0           1.5   

   signal_variance_standardized  noise_variance_standardized  log_marginal_likelihood  loo_rmse_absorbance  loo_mae_abs

## 2026-09-16 — corrected measured communities, four source rows

Append-only round 4. Earlier ODs, outputs and models above remain historical. The new loader uses full Echo reports for all three previous rounds, removes zero-volume species and regroups the measured wells by recorded composition and dose. It does not average broken intended triplicates. Empty inocula are excluded from fitting and the done registry. Replicate counts of 1 or 2 remain explicit.

EI and posterior-mean models each use their own corrected acquisition history plus corrected max-diversity history. Random stays outside training. The done registry uses measured recorded compositions, so intended-only failed communities can be selected again. Ordinary readouts, all-384-well plate-median subtraction and the existing presence-only GP remain unchanged. Dose variation is retained in lineage, not modeled as an extra feature. The September continuation report is used as the matching run record, with its inconsistent internal timestamps retained.

Source plate 396: **A1–A12 → plate 1 EI; B1–B12 → plate 2 max diversity; C1–C12 → plate 3 posterior mean; D1–D12 → plate 4 random.** Same column order in every row: **256, 75, 269, 294, 275, 41, 158, 145, 200, 196, 291, 264**. Species 296 excluded. Load **50 µL into each of the 48 wells**, with no refill between plates. Each species needs 200 µL working suspension across its four wells, excluding pipetting losses.

Today's OD field is the new `OD600_READING_AT_1_TO_10_20260916` block at the bottom of `adaptive_run.py`. Enter the 1:10 readings there; the code multiplies by 10. The cell below reloads those values. Missing OD means no executable Echo files. Keep 0.100 OD600·µL per species, 25 nL rounding, target working OD 1.20 where dilution is possible, 50 µL destination medium plus inoculum, three technical wells per community and no edge cultures. No 15 µL rule or automatic dose change. Source demand is summed over physical wells across all four picklists.

### Added validation repeats on plate 2
Reserve 24 wells: 294+196+291+264, 294+291+264, 294+275, and monocultures 294, 275, 196, 291, 264, each in triplicate. The remaining 282 culture wells hold 94 new max-diversity communities. Validation repeats are tagged separately from acquisition picks and use source row B.

The original September 4 four-member candidate lacked 196 in B23 but was complete in G16/L9. One 196 monoculture well (B22) also received zero inoculum. The 294+275 pair and its monocultures had full recorded transfers. Repeat the full and reduced community alongside their constituents. These same-plate comparisons keep the existing per-member dose; higher total biomass in mixtures remains a confound for any claim of synergy.


In [ ]:
from pathlib import Path
import sys, importlib
echo_dir_20260916 = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / 'glycerol_lactate_community/protocols/echo']
    if (candidate / 'acquisition_round_20260916.py').is_file())
if str(echo_dir_20260916) not in sys.path:
    sys.path.insert(0, str(echo_dir_20260916))
import adaptive_run, acquisition_round_20260916
importlib.reload(adaptive_run)
importlib.reload(acquisition_round_20260916)
# Edit today's OD block at the bottom of adaptive_run.py; old OD blocks stay unchanged.
round4_20260916 = adaptive_run.run_acquisition_round_20260916(base_dir=echo_dir_20260916)
display(round4_20260916['model_summary'])
display(round4_20260916['summary'])
display(round4_20260916['source_plan'])
display(round4_20260916['preparation_totals'])
if len(round4_20260916['source_demand']):
    display(round4_20260916['source_demand'])
print(round4_20260916['status'])
print('New snapshot:', round4_20260916['output_dir'])


## 2026-09-23 — one-plate analytical diagnostic round

Today: one culture plate, 12 monocultures + 41/200/196 + 256/269/294/41/145/200 + 294/275, each in six independently grown wells, plus six incubated medium controls. No acquisition picks or model retraining. Source plate 396 uses A1–A12 only, 50 µL each, in the established 12-species order. Enter fresh 1:10 readings in `OD600_READING_AT_1_TO_10_20260923` at the end of `adaptive_run.py`; missing ODs never create an executable Echo file.

Keep 50 µL culture medium plus inoculum, 0.100 OD600·µL per member, 25 nL rounding and the preceding incubation conditions/48 h endpoint. Measure endpoint OD600 before clearing. Earlier ODs and all preceding cells remain historical. Execute only this new section, not Run All.

At harvest: one combined assay session with one fresh reagent preparation. Each culture has two analytical aliquots, with duplicate 2 exactly one row below duplicate 1 for the VIAFLO 384 head: B/C, D/E, F/G, H/I, J/K, L/M, columns 2–17. Same 15 µL sample + 15 µL assay mix + 15 µL acid-stop volumes; do not halve the aliquots. The source map stays fixed for the second dispense. Verify loaded-tip/unused-channel geometry and shift clearance before transfer.

The planned assay includes 192 primary aliquots, 30 spike controls, 30 sample-background controls and 42 standards, all in the interior. Extra controls use culture replicate 1 of every biological condition. Confirmed on September 23: sample preparation is 5 µL supernatant + 10 µL fresh crude-glycerol screen medium, not water. Keep this unchanged. Medium standards use full-strength fresh medium; spike stock is also prepared in fresh screen medium so spiked and unspiked samples retain the same medium fractions. At 1:3, selected cultures require 30 µL original supernatant before handling losses, other cultures 10 µL. Recovered volume and instrument dead volume still need checking.

Read the complete bench instructions in [RUNBOOK_20260923_analytical.md](RUNBOOK_20260923_analytical.md). The generated session record holds actual timing, growth readout and Echo report references. These diagnostic observations remain outside training/acquisition scoring until reviewed.


In [ ]:
from pathlib import Path
import sys, importlib
echo_dir_20260923 = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / 'glycerol_lactate_community/protocols/echo']
    if (candidate / 'analytical_round_20260923.py').is_file())
if str(echo_dir_20260923) not in sys.path:
    sys.path.insert(0, str(echo_dir_20260923))
import adaptive_run, analytical_round_20260923
importlib.reload(adaptive_run)
importlib.reload(analytical_round_20260923)
# Today's ODs live only in the appended 20260923 block in adaptive_run.py.
diagnostic_20260923 = adaptive_run.run_analytical_round_20260923(base_dir=echo_dir_20260923)
print(diagnostic_20260923['status'])
display(diagnostic_20260923['source_plan'])
display(diagnostic_20260923['culture_design'])
display(diagnostic_20260923['sample_budget'])
if len(diagnostic_20260923['source_demand']):
    display(diagnostic_20260923['source_demand'])
print('Culture wells: 90 + 6 medium controls; assay wells: 294')
print('Full bench instructions:', echo_dir_20260923 / 'RUNBOOK_20260923_analytical.md')
print('New snapshot:', diagnostic_20260923['output_dir'])


## 2026-09-30 — execute the diagnostic plate with two source rows

Today's OD600 readings at 1:10 are appended in `OD600_READING_AT_1_TO_10_20260930` in `adaptive_run.py`. Same 15 diagnostic conditions, six culture replicates each, six medium controls and the accepted adjacent-row assay duplicate layout. Source rows A/B each contain the same 12 species in columns 1–12, 50 µL per usable well. **A4 must not be used. Every 294 transfer uses B4.** Replicates 1–3 otherwise use source A and 4–6 use B.

Working OD target 1.20; 269 and 196 remain undiluted at 0.80 and 1.10. Keep 0.100 OD600·µL per member, 25 nL rounding and 50 µL destination medium plus inoculum. Source demand and remainder are summed directly from the exported transfers. Execute only this dated cell for today's picklist. The existing diagnostic bench instructions apply, with the two-source-row/A4 exclusion overriding their old one-row instruction. No models are fitted for this plate.


In [ ]:
from pathlib import Path
import sys, importlib
echo_dir_20260930 = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / 'glycerol_lactate_community/protocols/echo']
    if (candidate / 'analytical_round_20260930.py').is_file())
if str(echo_dir_20260930) not in sys.path:
    sys.path.insert(0, str(echo_dir_20260930))
import adaptive_run, analytical_round_20260930
importlib.reload(adaptive_run)
importlib.reload(analytical_round_20260930)
diagnostic_20260930 = adaptive_run.run_analytical_round_20260930(base_dir=echo_dir_20260930)
print(diagnostic_20260930['status'])
display(diagnostic_20260930['source_plan'])
display(diagnostic_20260930['source_demand'])
display(diagnostic_20260930['well_audit'])
print('Echo file:', diagnostic_20260930['output_dir'] / 'plate-1_echo_picklist_20260930_analytical.csv')


## 2026-10-21 — small assay troubleshooting panel

Picks: 41, 200, 196 monocultures plus C01 (41 + 200 + 196), six culture wells each, plus six incubated medium blanks: 30 wells total. **196 uses the original stock**, not the poorly growing personal stock. No acquisitions or model fitting. Source plate 396: **A1=41, A2=200, A3=196; 50 µL total per well**. This is a new compact source map.

Enter fresh OD600 readings at 1:10 dilution in `OD600_READING_AT_1_TO_10_20261021` at the end of `adaptive_run.py`. All three start as None. This is the single editable OD input block; historical readings remain untouched. Pending ODs export maps only, never an executable Echo CSV. Keep 0.100 OD600·µL per member, 25 nL rounding, working OD target 1.20, 50 µL destination medium plus inoculum.

Culture rows B/D/F/H/J/L, columns 2–6. Each row mixes all four biological conditions and a blank; the generated CSV fixes their rotated order. Later assay: 60 primary aliquots, 10 matched spikes, 10 optical backgrounds and 24 water/medium standards = 104 wells. Main duplicates remain one row apart. Record endpoint OD, reaction pH checks on separate aliquots, and actual timing. Keep cultivation settings unchanged or document any seal change.

Read the appended **2026-10-21** section of [the bench runbook](RUNBOOK_20260923_analytical.md). Execute **only this dated section**, not Run All.


In [ ]:
from pathlib import Path
import sys, importlib
echo_dir_20261021 = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / 'glycerol_lactate_community/protocols/echo']
    if (candidate / 'analytical_round_20261021.py').is_file())
if str(echo_dir_20261021) not in sys.path:
    sys.path.insert(0, str(echo_dir_20261021))
import adaptive_run, analytical_round_20261021
importlib.reload(adaptive_run)
importlib.reload(analytical_round_20261021)
diagnostic_20261021 = adaptive_run.run_analytical_round_20261021(base_dir=echo_dir_20261021)
print(diagnostic_20261021['status'])
display(diagnostic_20261021['source_plan'])
display(diagnostic_20261021['culture_design'])
display(diagnostic_20261021['sample_budget'])
if len(diagnostic_20261021['source_demand']):
    display(diagnostic_20261021['source_demand'])
if diagnostic_20261021['ready']:
    print('Echo file:', diagnostic_20261021['output_dir'] / 'plate-1_echo_picklist_20261021_diagnostic.csv')
else:
    print('No executable Echo CSV. Enter or review the fresh ODs first.')
print('New snapshot:', diagnostic_20261021['output_dir'])
